Model Training

Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,OrdinalEncoder,StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error,mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV


In [ ]:
!pip install catboost

In [ ]:
from sklearn.linear_model import LinearRegression,Lasso,Ridge,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor,VotingRegressor,BaggingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
df=pd.read_csv('/content/MODEL_DATA.csv')
df.head()

,brand,model,color,year,power_ps,transmission_type,fuel_type,fuel_consumption_l_100km,fuel_consumption_g_km,mileage_in_km,price_in_euro
0,ford,Kuga,black,2023,190.0,Automatic,Hybrid,5.4,124.0,100.0,38490.0
1,hyundai,i10,black,2018,67.0,Manual,Petrol,4.6,106.0,27782.0,11555.0
2,audi,Q4 e-tron,grey,2021,170.0,Automatic,Electric,5.8,0.0,4247.0,48886.0
3,honda,CR-V,red,2018,155.0,Automatic,Petrol,7.5,175.0,57000.0,24490.0
4,kia,Sportage,black,2023,150.0,Manual,Petrol,5.9,150.0,7500.0,34990.0


In [ ]:
x=df.drop('price_in_euro',axis=1)
y=df['price_in_euro']

Split data into Dependent and Independent features

Split the dataset into training and testing sets.

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.30,random_state=30)

Encoding

In [ ]:
categorical_features_lb = ['fuel_type', 'brand', 'model', 'color']
categorical_features_one = ['transmission_type']
numeric_features = ['mileage_in_km', 'fuel_consumption_l_100km', 'fuel_consumption_g_km']

onehot_scaled = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

ordinal_scaled = Pipeline([
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ('scaler', StandardScaler())
])

numeric_scaled = Pipeline([
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot_scaled', onehot_scaled, categorical_features_one),
        ('ordinal_scaled', ordinal_scaled, categorical_features_lb),
        ('numeric_scaled', numeric_scaled, numeric_features)
    ]
)

X_train_encoded = preprocessor.fit_transform(x_train)
X_test_encoded = preprocessor.transform(x_test)

In [ ]:
X_train_encoded=pd.DataFrame(X_train_encoded)
X_test_encoded=pd.DataFrame(X_test_encoded)

Model Building

In [ ]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [ ]:
estimators=[('rf', RandomForestRegressor()), ('xg', XGBRegressor()), ('cat', CatBoostRegressor(verbose=False)), ('lgbm', LGBMRegressor())]

In [ ]:
models={

    'K-Neighbors Regressor':KNeighborsRegressor(),
    'SVR':SVR(kernel='linear'),#

    'LinearRegression':LinearRegression(),#
    'Lasso':Lasso(),#
    'Ridge':Ridge(),#
    'ElasticNet':ElasticNet(),#


    'Decision Tree':DecisionTreeRegressor(),

    'Random Forest Regressor':RandomForestRegressor(),
    'VotingRegressor':VotingRegressor(estimators=estimators),
    'BaggingRegressor':BaggingRegressor(estimator=LinearRegression(),bootstrap=True),

    'AdaBoost Regressor':AdaBoostRegressor(),#
    'GradientBoostingRegressor':GradientBoostingRegressor(),
    'XGBRegressor':XGBRegressor(),
    'CatBoosting Regressor':CatBoostRegressor(verbose=False),
    'LGBMRegressor':LGBMRegressor(),

    'StackingRegressor':StackingRegressor(estimators=estimators,final_estimator=XGBRegressor()),
}


In [ ]:
trained_model_list = []
model_list = []
r2_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train_encoded, y_train)

    # Train evaluation
    y_train_pred = model.predict(X_train_encoded)
    mae_train, rmse_train, r2_train = evaluate_model(y_train, y_train_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model Training Performance')
    print("RMSE:", rmse_train)
    print("MAE:", mae_train)
    print("R2 score:", r2_train * 100)

    # Test evaluation
    y_test_pred = model.predict(X_test_encoded)
    mae_test, rmse_test, r2_test = evaluate_model(y_test, y_test_pred)

    print('\nModel Testing Performance')
    print("RMSE:", rmse_test)
    print("MAE:", mae_test)
    print("R2 score:", r2_test * 100)

    r2_list.append(r2_test)  # save only test R² if you want comparison

    print('=' * 35)
    print('\n')


K-Neighbors Regressor
Model Training Performance
RMSE: 18569.797514846763
MAE: 6086.481868901018
R2 score: 82.77192421287239

Model Testing Performance
RMSE: 23378.400133742023
MAE: 7771.114508875739
R2 score: 74.63455201465655


SVR
Model Training Performance
RMSE: 41585.31307975268
MAE: 13018.384186719373
R2 score: 13.602258853111081

Model Testing Performance
RMSE: 43194.47862522574
MAE: 13540.81112204394
R2 score: 13.409662108535048


LinearRegression
Model Training Performance
RMSE: 37354.8552920095
MAE: 15736.90772721829
R2 score: 30.286553519692237

Model Testing Performance
RMSE: 38472.41859566397
MAE: 16037.335997867838
R2 score: 31.3070858780797




/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.183e+10, tolerance: 9.866e+09
  model = cd_fast.enet_coordinate_descent(


Lasso
Model Training Performance
RMSE: 37354.87275578222
MAE: 15735.698889403786
R2 score: 30.28648833619415

Model Testing Performance
RMSE: 38471.89338816336
MAE: 16036.146034206988
R2 score: 31.308961392476387


Ridge
Model Training Performance
RMSE: 37354.86388682407
MAE: 15736.447954228546
R2 score: 30.286521439567238

Model Testing Performance
RMSE: 38472.12667863847
MAE: 16036.836968182879
R2 score: 31.30812831605573


ElasticNet
Model Training Performance
RMSE: 38478.81484724519
MAE: 14972.871648305554
R2 score: 26.02826389777414

Model Testing Performance
RMSE: 39794.92506162893
MAE: 15336.446803957268
R2 score: 26.503214616572834


Decision Tree
Model Training Performance
RMSE: 1373.4804818844218
MAE: 94.03819199314839
R2 score: 99.9057530246246

Model Testing Performance
RMSE: 21796.25411204
MAE: 7077.18504369705
R2 score: 77.95161992282912


Random Forest Regressor
Model Training Performance
RMSE: 5857.991109808386
MAE: 2035.0848946281344
R2 score: 98.28556962088786

Model 

Hyper Parameter tuning

In [ ]:
final_models={

    'Random Forest Regressor':RandomForestRegressor(),
    'GradientBoostingRegressor':GradientBoostingRegressor(),
    'XGBRegressor':XGBRegressor(),
    'LGBMRegressor':LGBMRegressor(),
    'CatBoosting Regressor':CatBoostRegressor(verbose=False),
}

In [ ]:
param_grids = {
    'RandomForestRegressor': {
        'n_estimators': [100, 300, 500, 700],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [3, 5, 7, 10],
        'min_samples_leaf': [1, 2, 4],
        'min_samples_split': [2, 5, 10],
        'max_features': ['sqrt', 'log2', None],
    },

    'CatBoostRegressor': {
        'depth': [4, 6, 8],
        'iterations': [200, 500],
        'learning_rate': [0.01, 0.1, 0.2],
        'l2_leaf_reg': [3, 5, 6, 7, 8]
    },

    'XGBRegressor': {
        'n_estimators': [100, 200, 500],
        'learning_rate': [0.01, 0.1, 0.2, 0.05],
        'max_depth': [3, 5, 7],
        'colsample_bytree': [0.7, 0.5],
        'reg_alpha': [0.5, 1],
        'reg_lambda': [1.0, 0.5],
        'min_child_weight': [1, 2, 3]
    },

    'GradientBoostingRegressor': {
        'n_estimators': [100, 200, 500]
    },

    'LGBMRegressor': {
        'n_estimators': [100, 200, 500],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [5, 10, 15],
        'num_leaves': [31, 20, 15],
        'force_col_wise': [True, False],
        'reg_alpha': [0.5, 1],
        'reg_lambda': [0.5, 1],
        'min_child_samples': [5,10,15,20]
    }
}

In [ ]:
trained_model_list = []
model_list = []
r2_list = []

for model_name, model in final_models.items():
    print(f"Tuning {model_name}...")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grids.get(model_name, {}),
        n_iter=5,
        scoring='r2',
        cv=3,
        n_jobs=-1,
        random_state=42
    )

    # Fit with hyperparameter tuning
    search.fit(X_train_encoded, y_train)

    # Get best model
    best_model = search.best_estimator_
    trained_model_list.append(best_model)
    model_list.append(model_name)

    # Evaluate on train set
    y_train_pred = best_model.predict(X_train_encoded)
    mae_train, rmse_train, r2_train = evaluate_model(y_train, y_train_pred)

    # Evaluate on test set
    y_test_pred = best_model.predict(X_test_encoded)
    mae_test, rmse_test, r2_test = evaluate_model(y_test, y_test_pred)

    r2_list.append(r2_test)

    print(f"Best Params for {model_name}: {search.best_params_}")
    print(f"Train -> MAE: {mae_train:.4f}, RMSE: {rmse_train:.4f}, R2: {r2_train*100:.2f}%")
    print(f"Test  -> MAE: {mae_test:.4f}, RMSE: {rmse_test:.4f}, R2: {r2_test*100:.2f}%\n")


Tuning Random Forest Regressor...


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=5. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Params for Random Forest Regressor: {}
Train -> MAE: 2035.8227, RMSE: 5933.7819, R2: 98.24%
Test  -> MAE: 5466.7090, RMSE: 16970.9240, R2: 86.63%

Tuning GradientBoostingRegressor...


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=5. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Params for GradientBoostingRegressor: {'n_estimators': 500}
Train -> MAE: 6354.0375, RMSE: 14253.4189, R2: 89.85%
Test  -> MAE: 6868.5792, RMSE: 18627.0618, R2: 83.90%

Tuning XGBRegressor...
Best Params for XGBRegressor: {'reg_lambda': 1.0, 'reg_alpha': 0.5, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.5}
Train -> MAE: 4718.8000, RMSE: 10011.9747, R2: 94.99%
Test  -> MAE: 5761.9769, RMSE: 16640.2221, R2: 87.15%

Tuning LGBMRegressor...
[LightGBM] [Info] Total Bins 890
[LightGBM] [Info] Number of data points in the train set: 49291, number of used features: 11
[LightGBM] [Info] Start training from score 34968.575866
Best Params for LGBMRegressor: {'reg_lambda': 0.5, 'reg_alpha': 1, 'num_leaves': 15, 'n_estimators': 500, 'min_child_samples': 20, 'max_depth': 15, 'learning_rate': 0.2, 'force_col_wise': True}
Train -> MAE: 4800.7697, RMSE: 10492.7677, R2: 94.50%
Test  -> MAE: 5745.6771, RMSE: 16284.0121, R2: 87.69%

Tuning C

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=5. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Params for CatBoosting Regressor: {}
Train -> MAE: 5274.7727, RMSE: 10702.1275, R2: 94.28%
Test  -> MAE: 6001.2608, RMSE: 17290.8977, R2: 86.12%

